In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/Daily_Power_Gen_Source_march_23.csv")
df.head()

In [ ]:
df.shape


In [ ]:
df.columns


In [ ]:
df.info()

In [ ]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df.columns

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df.head()

In [ ]:
df_long = df.melt(
    id_vars=["date", "source"],
    value_vars=["nr", "wr", "sr", "er", "ner"],
    var_name="region",
    value_name="value"
)

df_long.head()

In [ ]:
df_long["source"] = df_long["source"].str.lower().str.strip()
df_long["region"] = df_long["region"].str.upper()

In [ ]:
df_final = df_long.pivot_table(
    index=["date", "region"],
    columns="source",
    values="value"
).reset_index()

df_final.head()

In [ ]:
df.head()

In [ ]:
df[df["source"].str.contains("Solar", case=False, na=False)].head(20)

In [ ]:
df_final.columns

In [ ]:
df_final.head(10)

In [ ]:
df_final.isnull().sum()

In [ ]:
df_final[df_final["solar gen (mu)"].notna()].head(10)

In [ ]:
df_final.columns = (
    df_final.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
)

In [ ]:
df_final.columns

In [ ]:
df_final.columns = (
    df_final.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("&", "and", regex=False)
)

df_final.columns

In [ ]:
df_final.columns.name = None
df_final.head()

In [ ]:
df_final[[
    "date",
    "region",
    "hydro",
    "total",
    "wind_genmu",
    "solar_gen_mu"
]].head(10)

In [ ]:
df_final[df_final["total"].isna()][["date", "region"]].head(20)

In [ ]:
for col in df_final.columns:
    if col not in ["date", "region"]:
        non_null = df_final[df_final[col].notna()]
        if len(non_null) > 0:
            print(f"{col}: starts from {non_null['date'].min().date()} | non-null rows = {len(non_null)}")
        else:
            print(f"{col}: no non-null values")

In [ ]:
df_final.shape

In [ ]:
df_final.head(10)

In [ ]:
df_final.tail(10)

In [ ]:
df_final.info()

In [ ]:
df_final.isnull().sum().sort_values(ascending=False)

In [ ]:
df_final.duplicated(subset=["date", "region"]).sum()

In [ ]:
df_final["region"].unique()

In [ ]:
df_final = df_final.sort_values(["date", "region"]).reset_index(drop=True)
df_final.to_csv("../data/processed/full_generation_pivot.csv", index=False)

In [ ]:
df_model = df_final.copy()
df_model["year"] = df_model["date"].dt.year
df_model["month"] = df_model["date"].dt.month
df_model["day"] = df_model["date"].dt.day
df_model = pd.get_dummies(df_model, columns=["region"], drop_first=True)
features = ["hydro", "wind_genmu", "year", "month", "day"] + \
           [col for col in df_model.columns if col.startswith("region_")]
df_model = df_model.dropna(subset=["total"])
X = df_model[features].fillna(0)
y = df_model["total"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)

In [ ]:
error = abs(y_test - y_pred)
threshold = error.mean()
reliability = (error < threshold).astype(int)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rel_model = RandomForestClassifier()
rel_model.fit(X_test, reliability)

In [ ]:
reliability_score = rel_model.predict_proba(X_test)[:, 1]

In [ ]:
safe_energy = y_pred * reliability_score
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "reliability": reliability_score,
    "safe_energy": safe_energy
})
results.head()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(results["predicted"], results["reliability"])
plt.xlabel("Predicted Power")
plt.ylabel("Reliability")
plt.title("Prediction vs Reliability")
plt.show()